[Project Stone]
- 돌 분류 프로젝트

In [1]:
# %pip install pandas

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

from torch.utils.data import Dataset, DataLoader  # Pytorch의 데이터셋 관련
from torchvision import transforms  # 전처리모듈
from torchvision.datasets import ImageFolder

from PIL import Image

import sys
import os

import pandas as pd
import numpy as np

In [3]:
print("--- Environment Check ---")
print(f"Python Executable: {sys.executable}") # 현재 사용 중인 파이썬 실행 파일 경로
print(f"Python Version: {sys.version}")      # 현재 사용 중인 파이썬 버전
print(f"PyTorch Version: {torch.__version__}") # 현재 로드된 PyTorch 버전
print(f"PyTorch CUDA Build: {torch.version.cuda if hasattr(torch.version, 'cuda') else 'N/A'}") # PyTorch가 빌드된 CUDA 버전
print(f"CUDA Available: {torch.cuda.is_available()}") # CUDA 사용 가능 여부
if torch.cuda.is_available():
    print(f"CUDA Version (Runtime): {torch.version.cuda}") # PyTorch가 인식하는 런타임 CUDA 버전
    print(f"Device Name: {torch.cuda.get_device_name(0)}") # GPU 이름
    print(f"Device Compute Capability: {torch.cuda.get_device_capability(0)}") # Compute Capability 확인
print("-" * 25)

--- Environment Check ---
Python Executable: c:\ProgramData\anaconda3\envs\DL_P310\python.exe
Python Version: 3.10.16 | packaged by Anaconda, Inc. | (main, Dec 11 2024, 16:19:12) [MSC v.1929 64 bit (AMD64)]
PyTorch Version: 2.4.0
PyTorch CUDA Build: 12.4
CUDA Available: True
CUDA Version (Runtime): 12.4
Device Name: NVIDIA A100 80GB PCIe
Device Compute Capability: (8, 0)
-------------------------


In [5]:
print(os.listdir('.'))
# os.mkdir('cs_stone')
# print(os.listdir('../../../../../../K/Desktop'))
# os.mkdir('../../../../../../K/Desktop/cs_stone')
cwd = '../../../../../../K/Desktop/cs_stone'

['_data']


In [10]:
numlist = os.listdir('./_data/open/train')
trainDIR = './_data/open/train'
sum = 0

for i in numlist:
    a = (len(os.listdir('./_data/open/train/'+i)))
    print(i, a)
    sum += a
print('train_sum', sum)

numlist = os.listdir('./_data/open/test')
testDIR = './_data/open/test'
print('test len', len(numlist))

Andesite 0
Basalt 8166
Etc 551
Gneiss 0
Granite 0
Mud_Sandstone 0
Weathered_Rock 0
train_sum 8717
test len 0


In [ ]:
## 데이터 불균형.

In [ ]:
TRANSFORM = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
])

TRANSFORM_MINOR = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(20),  # 더 강한 회전
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),  # 색상 다양화
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.8, 1.2)),  # 위치/스케일 다양화
    transforms.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 2.0)),  # 흐림 효과
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


In [ ]:
transform_dict = {
 'Andesite': TRANSFORM_MINOR,
 'Basalt': TRANSFORM_MINOR,
 'Gneiss': TRANSFORM,
 'Granite': TRANSFORM,
 'Mud_Sandstone': TRANSFORM,
 'Weathered_Rock': TRANSFORM_MINOR,
 'Etc': TRANSFORM_MINOR
}

In [ ]:
classes = ['Andesite', 'Basalt', 'Gneiss', 'Granite', 'Mud_Sandstone', 'Weathered_Rock']
classes2idx = {x:idx+1 for idx,x in enumerate(classes)}
classes2idx['Etc']= 0
classes2idx

{'Andesite': 1,
 'Basalt': 2,
 'Gneiss': 3,
 'Granite': 4,
 'Mud_Sandstone': 5,
 'Weathered_Rock': 6,
 'Etc': 0}

In [ ]:
class CustomImageFolder(ImageFolder):
    def __init__(self, root, transform_dict, classes2idx):
        self.transform_dict = transform_dict              # 클래스별 transform 저장
        self.classes2idx = classes2idx
        self.class_to_idx = classes2idx
        self.classes = list(classes2idx.keys())
        super().__init__(root, transform=None)            # transform은 직접 처리할 것이므로 None

    def find_classes(self, directory):
        # 고정된 클래스 사용
        return self.classes, self.classes2idx

    def __getitem__(self, index):
        path, target = self.samples[index]
        sample = Image.open(path).convert("RGB")

        # 클래스 인덱스를 다시 클래스명으로 매핑
        class_name = self.classes[target]
        transform = self.transform_dict.get(class_name)

        if transform:
            sample = transform(sample)

        return sample, target

In [ ]:
trainDS = CustomImageFolder(trainDIR, transform_dict, classes2idx=classes2idx)

In [ ]:
print('train', len(trainDS))
print(trainDS.classes)
trainDS.class_to_idx
idx2class = {values:key for key, values in classes2idx.items()}
idx2class

train 380020
['Andesite', 'Basalt', 'Gneiss', 'Granite', 'Mud_Sandstone', 'Weathered_Rock', 'Etc']


{1: 'Andesite',
 2: 'Basalt',
 3: 'Gneiss',
 4: 'Granite',
 5: 'Mud_Sandstone',
 6: 'Weathered_Rock',
 0: 'Etc'}

In [ ]:
from torch.utils.data import random_split

# 전체 데이터 수
total_size = len(trainDS)
print(total_size)
train_size = int(0.8 * total_size)
valid_size = total_size - train_size

# 무작위로 train/valid 분리
# trainDS 304016
# validDS 76004
trainDS, validDS = random_split(trainDS, [train_size, valid_size])
print(len(trainDS),len(validDS))

380020
304016 76004


In [ ]:
def collator(batch):
    images, labels = zip(*batch)  # 튜플 of Tensors
    images = torch.stack(images)  # → Tensor of shape [B, C, H, W]
    labels = torch.tensor(labels) # → Tensor of shape [B]
    return images, labels

BATCH_SIZE = 100
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LR = 0.0001
NUM_CLASSES = len(classes2idx)
print(DEVICE)

cpu


In [ ]:
trainDL = DataLoader(
    trainDS, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, collate_fn=collator
)
validDL = DataLoader(
    validDS, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, collate_fn=collator
)

In [ ]:
from torchvision import models
from torchvision import ops
from torchvision.models.detection import rpn
from torch import optim
from tqdm import tqdm


In [ ]:
from torchvision.models import (
    ResNet50_Weights, ResNet101_Weights, ResNet152_Weights, # 예시 ResNet 가중치
    Swin_T_Weights, Swin_S_Weights, Swin_B_Weights,       # 예시 Swin Transformer 가중치
    EfficientNet_B0_Weights, EfficientNet_B4_Weights, EfficientNet_B7_Weights # 예시 EfficientNet 가중치
)

In [ ]:
def _load_pretrained_model(model_name, weights, num_classes):
    """지정된 모델과 가중치를 로드하고 분류 레이어를 수정합니다."""
    model = model_name(weights=weights)

    if isinstance(model, models.ResNet):
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
    elif isinstance(model, models.SwinTransformer):
        in_features = model.head.in_features
        model.head = nn.Linear(in_features, num_classes)
    elif isinstance(model, models.EfficientNet):
        # EfficientNet의 classifier는 보통 (dropout, linear) 형태의 Sequential
        in_features = model.classifier[1].in_features
        # 기존 dropout 설정을 유지하거나 새로 정의할 수 있습니다.
        dropout_p = model.classifier[0].p if hasattr(model.classifier[0], 'p') else 0.2
        model.classifier = nn.Sequential(
            nn.Dropout(p=dropout_p, inplace=True),
            nn.Linear(in_features, num_classes)
        )
    else:
        raise TypeError(f"Unsupported model type: {type(model)}")

    return model


In [ ]:
class ResNetSwinEfficientNetEnsemble(nn.Module):
    """
    ResNet, Swin Transformer, EfficientNet 모델의 예측 로짓(logits)을 평균 내는 앙상블 모델.
    (이전 버전은 확률을 평균냈음)
    """
    def __init__(self, num_classes: int, resnet_type: str = 'resnet50', swin_type: str = 'swin_t', effnet_type: str = 'efficientnet_b0'):
        super().__init__()
        self.num_classes = num_classes

        # --- 모델 및 가중치 선택 (이전 코드와 동일) ---
        # ResNet
        if resnet_type == 'resnet50':
            resnet_builder = models.resnet50
            resnet_weights = models.ResNet50_Weights.IMAGENET1K_V1
        elif resnet_type == 'resnet101':
            resnet_builder = models.resnet101
            resnet_weights = models.ResNet101_Weights.IMAGENET1K_V1
        else: raise ValueError(f"Unsupported resnet_type: {resnet_type}")
        # Swin Transformer
        if swin_type == 'swin_t':
            swin_builder = models.swin_t
            swin_weights = models.Swin_T_Weights.IMAGENET1K_V1
        elif swin_type == 'swin_s':
            swin_builder = models.swin_s
            swin_weights = models.Swin_S_Weights.IMAGENET1K_V1
        elif swin_type == 'swin_b':
             swin_builder = models.swin_b
             swin_weights = models.Swin_B_Weights.IMAGENET1K_V1
        else: raise ValueError(f"Unsupported swin_type: {swin_type}")
        # EfficientNet
        if effnet_type == 'efficientnet_b0':
            effnet_builder = models.efficientnet_b0
            effnet_weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
        elif effnet_type == 'efficientnet_b4':
            effnet_builder = models.efficientnet_b4
            effnet_weights = models.EfficientNet_B4_Weights.IMAGENET1K_V1
        else: raise ValueError(f"Unsupported effnet_type: {effnet_type}")

        # --- 개별 모델 로드 및 수정 (이전 코드와 동일) ---
        self.model_resnet = _load_pretrained_model(resnet_builder, resnet_weights, num_classes)
        self.model_swin = _load_pretrained_model(swin_builder, swin_weights, num_classes)
        self.model_effnet = _load_pretrained_model(effnet_builder, effnet_weights, num_classes)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        앙상블 모델의 forward pass를 수행하여 평균 로짓을 반환합니다.

        Args:
            x (torch.Tensor): 입력 이미지 배치 (B, C, H, W).

        Returns:
            torch.Tensor: 각 클래스에 대한 평균 예측 로짓 (B, num_classes).
        """
        # 각 모델로부터 로짓(logits) 얻기
        logits_resnet = self.model_resnet(x)
        logits_swin = self.model_swin(x)
        logits_effnet = self.model_effnet(x)

        # 로짓들을 쌓음 (새로운 차원 생성: [3, Batch_size, Num_classes])
        all_logits = torch.stack([logits_resnet, logits_swin, logits_effnet], dim=0)

        # 모델 차원(dim=0)에 대해 평균을 내어 최종 앙상블 로짓 계산
        avg_logits = torch.mean(all_logits, dim=0)

        return avg_logits # 평균 로짓 반환

In [ ]:
# ResNet101
# Swin Transformer - resnet과 다른 접근방식. attention 매커니즘 사용
# EfficientNet - cnn계열이지만 다른 방식으로 접근함.
# 추가고려 - Densenet
model = ResNetSwinEfficientNetEnsemble(
        num_classes=NUM_CLASSES,
        resnet_type='resnet101',
        swin_type='swin_t',
        effnet_type='efficientnet_b0'
    ).to(DEVICE)

Downloading: "https://download.pytorch.org/models/swin_t-704ceda3.pth" to C:\Users\kdt/.cache\torch\hub\checkpoints\swin_t-704ceda3.pth
100%|██████████| 108M/108M [00:02<00:00, 42.8MB/s] 
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\kdt/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:00<00:00, 30.6MB/s]


In [ ]:
# 손실함수 FocalLoss
# 데이터 불균형 존재
from typing import Union # Union 임포트 추가

class FocalLoss(nn.Module):
    """
    Focal Loss 구현. Multi-class 분류 문제에 사용.
    논문: Lin et al., Focal Loss for Dense Object Detection (https://arxiv.org/abs/1708.02002)

    Args:
        alpha (float or Tensor, optional): 클래스별 가중치. 불균형이 심할 때 소수 클래스에 높은 가중치를 줄 수 있음.
                                            float이면 모든 클래스에 동일하게 적용 (보통 0.25 추천).
                                            Tensor이면 클래스 수와 동일한 길이의 1D 텐서. Defaults to 0.25.
        gamma (float, optional): Focusing 파라미터. 클수록 쉬운 샘플의 손실을 더 많이 줄임. Defaults to 2.0.
        reduction (str, optional): 손실 감소 방식: 'mean', 'sum', 'none'. Defaults to 'mean'.
    """
    # def __init__(self, alpha: float | torch.Tensor | None = 0.25, gamma: float = 2.0, reduction: str = 'mean'):
    ## python 3.9이하이므로 | 대신 Union사용.
    def __init__(self, alpha: Union[float, torch.Tensor, None] = 0.25, gamma: float = 2.0, reduction: str = 'mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        # alpha를 버퍼로 등록하여 .to(device) 호출 시 자동으로 이동되도록 함
        if alpha is not None:
            if isinstance(alpha, float):
                # 모든 클래스에 동일한 alpha 적용 (internal broadcasting)
                self.register_buffer('alpha', torch.tensor([alpha]))
            elif isinstance(alpha, torch.Tensor):
                if alpha.ndim > 1 or alpha.numel() == 0 : # 스칼라 텐서 또는 1D 텐서여야 함
                     raise ValueError("alpha tensor must be a scalar or a 1D tensor with length equal to num_classes")
                self.register_buffer('alpha', alpha) # 클래스별 alpha
            else:
                raise TypeError(f"Unsupported alpha type: {type(alpha)}")
        else:
            self.register_buffer('alpha', None)

        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Args:
            logits (torch.Tensor): 모델의 예측 로짓 (B, C).
            targets (torch.Tensor): 실제 클래스 레이블 (B). 정수형 타입이어야 함.

        Returns:
            torch.Tensor: 계산된 Focal Loss. 스칼라 값 (reduction='mean' or 'sum').
        """
        num_classes = logits.shape[1]
        targets = targets.long() # 인덱싱을 위해 long 타입 확인

        # log_softmax 계산 (수치 안정성)
        log_probs = F.log_softmax(logits, dim=1)

        # 실제 클래스에 해당하는 log_probs 값 선택
        log_pt = log_probs.gather(1, targets.view(-1, 1)).squeeze(1) # (B)

        # 실제 클래스에 해당하는 확률 pt 계산
        pt = log_pt.exp() # (B)

        # alpha 가중치 계산
        if self.alpha is not None:
            if self.alpha.numel() == 1: # 스칼라 alpha
                alpha_t = self.alpha.repeat(targets.shape[0])
            else: # 클래스별 alpha
                 # alpha 텐서의 길이가 num_classes와 같은지 확인 (초기화 시 확인하는 것이 더 좋음)
                 if self.alpha.numel() != num_classes:
                     raise ValueError(f"Length of alpha tensor ({self.alpha.numel()}) must match num_classes ({num_classes})")
                 alpha_t = self.alpha.gather(0, targets) # 각 샘플의 타겟 클래스에 맞는 alpha 값 선택 (B)
        else:
            # alpha를 사용하지 않으면 가중치는 1
            alpha_t = torch.ones_like(targets, dtype=torch.float, device=logits.device)

        # Focal Loss 계산: -alpha * (1 - pt)^gamma * log(pt)
        loss = -alpha_t * torch.pow(1.0 - pt, self.gamma) * log_pt

        # Reduction 적용
        if self.reduction == 'mean':
            loss = loss.mean()
        elif self.reduction == 'sum':
            loss = loss.sum()
        elif self.reduction == 'none':
            pass # 개별 손실 반환
        else:
            raise ValueError(f"Invalid reduction mode: {self.reduction}")

        return loss

In [ ]:
# ✅ 손실함수, 옵티마이저
# criterion = nn.CrossEntropyLoss()
criterion = FocalLoss()
optimizer = optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=0.01
    # betas=(0.9, 0.999), # 기본값 사용 가능
    # eps=1e-8           # 기본값 사용 가능
)



In [ ]:
# %pip install psutil
import psutil
import subprocess
from tqdm import tqdm

def get_gpu_usage():
    try:
        result = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv,nounits,noheader"]
        )
        result = result.decode("utf-8").strip().split("\n")[0]
        gpu_util, mem_used, mem_total = map(int, result.split(", "))
        return gpu_util, mem_used, mem_total
    except Exception:
        return None, None, None

In [ ]:
import time

In [ ]:
now = time.localtime()
ct = time.strftime("%y.%m.%d %H:%M:%S",now)
ct

'25.04.30 15:48:55'

In [ ]:
EPOCH = 5
PATIENCE = 3

In [ ]:
print(f"Training started on device: {DEVICE}")
print(f"Model: Ensemble(ResNet, Swin, EfficientNet), Loss: Focal Loss, Optimizer: AdamW")

best_val_loss = float('inf')
patience_counter = 0

# 에포크별 '평균' 지표를 저장할 리스트
train_metrics = []
val_metrics = []

# psutil.cpu_percent() 초기 호출 (첫 호출 시 의미 없는 값 반환 방지)
psutil.cpu_percent(interval=None)
time.sleep(0.1) # CPU 사용률 측정을 위한 짧은 대기

for epoch in range(EPOCH):
    # ----------- Training -----------
    model.train()
    # --- 에포크 누적 변수 초기화 ---
    epoch_train_loss_sum = 0.0
    epoch_train_correct = 0
    epoch_train_total = 0
    epoch_train_cpu_sum = 0.0
    epoch_train_gpu_sum = 0.0
    train_batches = 0 # 실제 처리된 배치 수 카운트

    train_loop = tqdm(trainDL, desc=f"[Epoch {epoch+1}/{EPOCH}] Training", leave=False)
    for images, labels in train_loop:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        # --- 순전파 및 손실 계산 ---
        outputs = model(images) # 모델은 이제 로짓(logits)을 반환
        loss = criterion(outputs, labels) # Focal Loss 사용

        # --- 역전파 및 최적화 ---
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # --- 배치 결과 누적 ---
        epoch_train_loss_sum += loss.item()
        _, predicted = outputs.max(1) # 로짓에서 바로 argmax 사용 가능
        epoch_train_correct += predicted.eq(labels).sum().item()
        epoch_train_total += labels.size(0)
        train_batches += 1

        # --- 시스템 지표 측정 및 누적 ---
        cpu_usage = psutil.cpu_percent(interval=None)
        gpu_util, _, _ = get_gpu_usage() # 메모리 사용량은 여기선 저장 안 함
        epoch_train_cpu_sum += cpu_usage
        epoch_train_gpu_sum += gpu_util if gpu_util is not None else 0

        # --- tqdm 진행률 표시줄 업데이트 ---
        if epoch_train_total > 0:
             current_cumulative_acc = 100. * epoch_train_correct / epoch_train_total
        else:
             current_cumulative_acc = 0
        train_loop.set_postfix(loss=loss.item(),
                               acc=f"{current_cumulative_acc:.2f}%",
                               cpu=f"{cpu_usage:.1f}%",
                               gpu=f"{gpu_util if gpu_util is not None else 0:.1f}%")

    # --- 에포크 평균 계산 ---
    if train_batches > 0:
        avg_train_loss = epoch_train_loss_sum / train_batches
        avg_train_acc = 100. * epoch_train_correct / epoch_train_total
        avg_train_cpu = epoch_train_cpu_sum / train_batches
        avg_train_gpu = epoch_train_gpu_sum / train_batches
    else:
        avg_train_loss, avg_train_acc, avg_train_cpu, avg_train_gpu = 0, 0, 0, 0

    # --- 에포크 평균 지표 저장 ---
    train_metrics.append({
        "epoch": epoch + 1,
        "loss": avg_train_loss,
        "accuracy": avg_train_acc,
        "avg_cpu_percent": avg_train_cpu,
        "avg_gpu_percent": avg_train_gpu
    })

    # ----------- Validation -----------
    model.eval()
    # --- 에포크 누적 변수 초기화 ---
    epoch_val_loss_sum = 0.0
    epoch_val_correct = 0
    epoch_val_total = 0
    epoch_val_cpu_sum = 0.0
    epoch_val_gpu_sum = 0.0
    val_batches = 0 # 실제 처리된 배치 수 카운트

    val_loop = tqdm(validDL, desc=f"[Epoch {epoch+1}/{EPOCH}] Validation", leave=False)
    with torch.no_grad():
        for images, labels in val_loop:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            outputs = model(images) # 로짓 반환
            loss = criterion(outputs, labels) # Focal Loss 사용

            # --- 배치 결과 누적 ---
            epoch_val_loss_sum += loss.item()
            _, predicted = outputs.max(1)
            epoch_val_correct += predicted.eq(labels).sum().item()
            epoch_val_total += labels.size(0)
            val_batches += 1

            # --- 시스템 지표 측정 및 누적 ---
            cpu_usage = psutil.cpu_percent(interval=None)
            gpu_util, _, _ = get_gpu_usage()
            epoch_val_cpu_sum += cpu_usage
            epoch_val_gpu_sum += gpu_util if gpu_util is not None else 0

            # --- tqdm 진행률 표시줄 업데이트 ---
            if epoch_val_total > 0:
                current_cumulative_acc = 100. * epoch_val_correct / epoch_val_total
            else:
                current_cumulative_acc = 0
            val_loop.set_postfix(loss=loss.item(),
                                 acc=f"{current_cumulative_acc:.2f}%",
                                 cpu=f"{cpu_usage:.1f}%",
                                 gpu=f"{gpu_util if gpu_util is not None else 0:.1f}%")

    # --- 에포크 평균 계산 ---
    if val_batches > 0:
        avg_val_loss = epoch_val_loss_sum / val_batches
        avg_val_acc = 100. * epoch_val_correct / epoch_val_total
        avg_val_cpu = epoch_val_cpu_sum / val_batches
        avg_val_gpu = epoch_val_gpu_sum / val_batches
    else:
        avg_val_loss, avg_val_acc, avg_val_cpu, avg_val_gpu = 0, 0, 0, 0

    # --- 에포크 평균 지표 저장 ---
    val_metrics.append({
        "epoch": epoch + 1,
        "loss": avg_val_loss,
        "accuracy": avg_val_acc,
        "avg_cpu_percent": avg_val_cpu,
        "avg_gpu_percent": avg_val_gpu
    })

    # --- 결과 출력 및 모델 저장 로직 (Early Stopping 포함 가능) ---
    print(f"\n[Epoch {epoch+1}/{EPOCH}]")
    print(f"  Train Loss: {avg_train_loss:.4f}, Train Acc: {avg_train_acc:.2f}% | Avg CPU: {avg_train_cpu:.1f}%, Avg GPU: {avg_train_gpu:.1f}%")
    print(f"  Valid Loss: {avg_val_loss:.4f}, Valid Acc: {avg_val_acc:.2f}% | Avg CPU: {avg_val_cpu:.1f}%, Avg GPU: {avg_val_gpu:.1f}%")

    # Early Stopping 및 모델 저장 (평균 검증 손실 사용)
    if avg_val_loss < best_val_loss :
        print(f"  Validation loss improved ({best_val_loss:.4f} --> {avg_val_loss:.4f}). Saving model...")
        best_val_loss = avg_val_loss
        # 모델 저장 (state_dict 저장 권장)
        try:
            # _model 폴더가 없다면 생성하거나 다른 경로 지정
            torch.save(model.state_dict(), f"./epoch{epoch+1}_best_vl{avg_val_loss:.4f}.pth")
        except FileNotFoundError:
            print("  Warning: Directory not found. Saving model to current directory.")
            torch.save(model.state_dict(), f"./epoch{epoch+1}_best_vl{avg_val_loss:.4f}.pth")

        patience_counter = 0 # 개선되었으므로 카운터 초기화
    else:
        patience_counter += 1
        print(f"  Validation loss did not improve from {best_val_loss:.4f}. Patience: {patience_counter}/{PATIENCE}")
        if patience_counter >= PATIENCE:
            print(f"  Early stopping triggered after {epoch + 1} epochs.")
            break # 학습 중단


Training started on device: cpu
Model: Ensemble(ResNet, Swin, EfficientNet), Loss: Focal Loss, Optimizer: AdamW


[Epoch 1/5] Training:   0%|          | 4/3040 [12:50<164:35:52, 195.18s/it, acc=41.50%, cpu=52.3%, gpu=0.0%, loss=0.24] 

In [ ]:
# 이제 이 데이터를 pandas DataFrame으로 변환하여 CSV로 저장할 수 있습니다.
# 예시:
train_df = pd.DataFrame(train_metrics)
val_df = pd.DataFrame(val_metrics)
train_df.to_csv("train_metrics.csv", index=False)
val_df.to_csv("val_metrics.csv", index=False)

NameError: name 'train_metrics' is not defined

In [ ]:
sampleDF = pd.DataFrame(pd.read_csv('./_data/open/sample_submission.csv'))
print(sampleDF.head())
testDF = pd.DataFrame(pd.read_csv('./_data/open/test.csv'))
print(testDF.head())

           ID rock_type
0  TEST_00000       Etc
1  TEST_00001       Etc
2  TEST_00002       Etc
3  TEST_00003       Etc
4  TEST_00004       Etc
           ID               img_path
0  TEST_00000  ./test/TEST_00000.jpg
1  TEST_00001  ./test/TEST_00001.jpg
2  TEST_00002  ./test/TEST_00002.jpg
3  TEST_00003  ./test/TEST_00003.jpg
4  TEST_00004  ./test/TEST_00004.jpg


In [ ]:
class TestImageDataset(Dataset):
    def __init__(self, csv_df, image_root, transform=None):
        self.df = csv_df
        self.image_root = image_root  # 예: './_data/open/test/'
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx, 0]
        img_path = self.df.iloc[idx, 1]
        img_path = os.path.join(self.image_root, img_path)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, img_name


In [ ]:
TRANSFORM_T = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
])



In [ ]:
testDS = TestImageDataset(testDF, image_root="./_data/open/", transform=TRANSFORM_T)
testDL = DataLoader(testDS, batch_size=32, shuffle=False)

In [ ]:
model.eval()
predictions = []

with torch.no_grad():
    count = 0
    for images, filenames in testDL:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        
        for fname, pred in zip(filenames, preds):
            predictions.append((fname, idx2class[pred.item()]))
        count += 1
        # if count ==3: break

In [ ]:
sampleDF.head()

,ID,rock_type
0,TEST_00000,Etc
1,TEST_00001,Etc
2,TEST_00002,Etc
3,TEST_00003,Etc
4,TEST_00004,Etc


In [1]:
len(predictions)

NameError: name 'predictions' is not defined

In [ ]:
# for i in range(sampleDF.shape[0]):
for i in range(95006):
    if sampleDF.loc[i,'ID'] == predictions[i][0]:
        sampleDF.loc[i,'rock_type'] = predictions[i][1]
    else:
        continue
sampleDF.head(20)

IndexError: list index out of range

In [ ]:
sampleDF.to_csv('./_data/open/sample_submission_answer.csv', index=False)